**Connect to Google Drive first, before anything else.** Free Colab runtimes disconnect unpredictably, and steps 1/2 below (synthetic clip generation + augmentation) can take a long time at production sample counts. Pointing `output_dir` at a mounted Drive folder means those generated clips -- and the final exported model -- survive a disconnect instead of vanishing with the ephemeral Colab VM. (This does *not* checkpoint an in-progress **training run** itself -- `train.py`'s auto-training keeps candidate checkpoints in memory only, so a disconnect during Step 3 still loses that run; you'd rerun Step 3 from the already-preserved clips/features, not from scratch.) Mounting first also means the Drive auth prompt doesn't interrupt you partway through the environment setup / installs below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup (minimal -- just enough for the preflight check below)

import os

# install piper-sample-generator -- via the actively-maintained PyPI package,
# not a git clone + sys.path hack. The notebook used to clone
# rhasspy/piper-sample-generator and import it as a bare module, but that
# repo restructured its generate_samples() API into a proper package in 2026
# ("Move to package") and dropped the old piper-phonemize C-extension in
# favor of a self-contained espeak-ng bridge bundled in piper-tts. train.py
# is patched a few cells down to match this current package instead of
# reverting to the old, no-longer-compatible API.
#
# Deliberately NOT installing openwakeword or the other training/augmentation
# dependencies yet -- those are heavier (some pull in several hundred MB of
# their own deps) and aren't needed to find out whether piper_sample_generator
# and its checkpoint even work. That happens after the preflight check below.
!pip install piper-sample-generator==3.2.0
!pip install deep-phonemizer==0.0.19  # needed by openwakeword's adversarial-text generation; installed here (not with the other heavy deps below) so the preflight check can exercise it too
os.makedirs("piper-sample-generator/models", exist_ok=True)
!wget -nc -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# The GitHub release above only ships the .pt weights, not the .pt.json config
# (voice, sample rate, phoneme_id_map, num_speakers) that generate_samples()
# needs to load them -- that file only exists checked into git, not in the
# release assets. Fetch it from the same pinned commit used for piper_train
# above, so it matches this exact model.
!wget -nc -O piper-sample-generator/models/en_US-libritts_r-medium.pt.json 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/2971426a55072f7d22fec416ca7800df8bd23207/models/en_US-libritts_r-medium.pt.json'


**Fetch the missing `piper_train` package.** `piper-sample-generator==3.2.0`'s `__main__.py` needs `piper_train.vits.commons` at import time, and `piper_train.vits.models` when it later calls `torch.load()` on a `.pt` generator checkpoint -- the pickle embeds class references like `SynthesizerTrn`, so unpickling requires the exact matching class definitions to be importable. `piper_train` actually lives in the same MIT-licensed `rhasspy/piper-sample-generator` repo as `piper_sample_generator` (not a separate GPL project), but that repo's `pyproject.toml` only packages `piper_sample_generator*` -- so the PyPI wheel silently drops `piper_train`, even though `__main__.py` needs it. Fetch the handful of files it actually needs (`commons.py`, `attentions.py`, `modules.py`, `transforms.py`, `models.py`) straight from the git tag matching the installed version (`v3.2.0`) and put them on `PYTHONPATH`, so the source matches exactly what the checkpoint was built against.

In [ ]:
import os
import urllib.request

# Pinned to the git tag matching piper-sample-generator==3.2.0 above, so the
# fetched piper_train source matches exactly what the checkpoint expects.
PIPER_TRAIN_COMMIT = "2971426a55072f7d22fec416ca7800df8bd23207"  # tag v3.2.0
RAW_BASE = (
    f"https://raw.githubusercontent.com/rhasspy/piper-sample-generator/"
    f"{PIPER_TRAIN_COMMIT}/piper_train"
)

vits_dir = "piper_train_stub/piper_train/vits"
os.makedirs(vits_dir, exist_ok=True)
open("piper_train_stub/piper_train/__init__.py", "w").close()

for name in [
    "__init__.py",
    "commons.py",
    "attentions.py",
    "modules.py",
    "transforms.py",
    "models.py",
]:
    urllib.request.urlretrieve(f"{RAW_BASE}/vits/{name}", os.path.join(vits_dir, name))

# Steps 1-3 (and the preflight check below) run train.py / generate_samples as
# subprocesses (`!{sys.executable} ...`), so the fetched package has to reach
# them via PYTHONPATH rather than sys.path.
stub_path = os.path.abspath("piper_train_stub")
os.environ["PYTHONPATH"] = stub_path + os.pathsep + os.environ.get("PYTHONPATH", "")


**Patch `deep-phonemizer`'s `torch.load()` call.** `deep-phonemizer==0.0.19` (latest on PyPI, no newer release exists) calls `torch.load(checkpoint_path, map_location=device)` in `dp/model/model.py` without `weights_only=False`. PyTorch >=2.6 defaults `weights_only=True`, which refuses to unpickle the checkpoint's custom classes (e.g. `dp.preprocessing.text.Preprocessor`) and raises `UnpicklingError`. openwakeword's `generate_adversarial_texts()` (used by Step 1 for out-of-vocabulary words) hits this via `Phonemizer.from_checkpoint()`. Patch it the same way `train.py` is patched below.

In [ ]:
import dp

dp_model_path = os.path.join(os.path.dirname(dp.__file__), "model", "model.py")
with open(dp_model_path) as f:
    dp_model_src = f.read()

old = "checkpoint = torch.load(checkpoint_path, map_location=device)"
new = "checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)"
if old in dp_model_src:
    with open(dp_model_path, "w") as f:
        f.write(dp_model_src.replace(old, new))
    print("Patched", dp_model_path)
else:
    print("Already patched (or upstream changed), skipping")


**Preflight: fail fast on import/compatibility issues.** Only `piper-sample-generator` and `deep-phonemizer` (plus their checkpoints) are installed so far -- deliberately, so this can run before the heavier installs (openwakeword, speechbrain, datasets, etc.) and the multi-GB dataset downloads later in the notebook. Generate a single throwaway sample, and separately run the DeepPhonemizer checkpoint on one out-of-vocabulary word, in a fresh subprocess with the same environment/`PYTHONPATH` those later steps will use. Together these exercise the full chain Step 1 needs end-to-end -- `piper_train` imports, unpickling both `.pt` checkpoints, the VITS forward pass, the espeak-ng phonemizer, and the DeepPhonemizer forward pass -- so any incompatibility surfaces in seconds, before paying for installs and downloads that would turn out not to matter, or for Step 1's own ~10 minute run.

In [ ]:
import os
import subprocess
import sys
import tempfile
import urllib.request

with tempfile.TemporaryDirectory() as smoke_dir:
    smoke_script = (
        "from piper_sample_generator.__main__ import generate_samples\n"
        "generate_samples(\n"
        "    text=['testing one two three'],\n"
        f"    output_dir={smoke_dir!r},\n"
        "    model='piper-sample-generator/models/en_US-libritts_r-medium.pt',\n"
        "    max_samples=1,\n"
        ")\n"
    )
    result = subprocess.run(
        [sys.executable, "-c", smoke_script],
        capture_output=True,
        text=True,
    )
    print(result.stdout[-2000:])
    print(result.stderr[-4000:])
    generated = os.listdir(smoke_dir)

assert result.returncode == 0, (
    "Preflight sample-generation check failed -- fix this before installing "
    "the rest of the (heavier) dependencies below."
)
assert generated, "Preflight check produced no audio file."
print(f"Preflight OK: generated {generated} in a subprocess using the same PYTHONPATH later steps will use.")

# Same checkpoint URL/path openwakeword's generate_adversarial_texts() uses --
# fetch once and cache, so this and Step 1 later both reuse it.
phonemizer_mdl_path = "en_us_cmudict_forward.pt"
if not os.path.exists(phonemizer_mdl_path):
    urllib.request.urlretrieve(
        "https://public-asai-dl-models.s3.eu-central-1.amazonaws.com/DeepPhonemizer/en_us_cmudict_forward.pt",
        phonemizer_mdl_path,
    )

phonemizer_script = (
    "from dp.phonemizer import Phonemizer\n"
    f"phonemizer = Phonemizer.from_checkpoint({phonemizer_mdl_path!r})\n"
    "print(phonemizer('wombat', lang='en_us'))\n"
)
result = subprocess.run(
    [sys.executable, "-c", phonemizer_script],
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:])
print(result.stderr[-4000:])
assert result.returncode == 0, (
    "Preflight DeepPhonemizer check failed -- fix this before installing "
    "the rest of the (heavier) dependencies below."
)
print("Preflight OK: DeepPhonemizer checkpoint loaded and ran in a subprocess.")


**Install the rest of the (heavier) dependencies.** The preflight checks above passed, so it's worth installing openwakeword and the augmentation/training packages now -- `speechbrain`, `datasets`, etc. -- plus the pre-trained embedding/melspectrogram models openwakeword needs for feature extraction.

In [ ]:
import os

# install openwakeword (full installation to support training)
if not os.path.exists("openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# install other dependencies
# (dropped tensorflow-cpu / tensorflow_probability / onnx_tf here -- only needed
# for the optional .tflite export in the old "retry tflite" cell, which this
# repo never uses; wake_word_daemon.py loads the model with inference_framework="onnx")
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.12.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
# deep-phonemizer already installed above (needed by the preflight check)

# Download required models (workaround for Colab)
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -nc https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


**Pin `torch`/`torchaudio`/`torchvision` for `torch-audiomentations` compatibility.** TorchAudio deprecated its legacy I/O dispatcher (`torchaudio.info`/`load`/`save`) in 2.8 and removed it in 2.9, consolidating into TorchCodec. `torch-audiomentations==0.12.0` (the latest release, installed above) still calls `torchaudio.info()` directly in its background-noise augmenter, so on Colab's preinstalled torch/torchaudio (2.9.x) Step 2 (`--augment_clips`) fails with `AttributeError: module 'torchaudio' has no attribute 'info'`. Pin torch/torchaudio to the last pair that still has it (2.8.0), installed last in this cell so it isn't silently upgraded back by another package's dependency resolution. `torchvision` has to be pinned in lockstep (0.23.0, the release built against torch 2.8.0) -- its compiled ops (e.g. `torchvision::nms`, used transitively by `torchmetrics` at import time) are ABI-locked to one specific torch build, so leaving Colab's preinstalled torchvision (built for 2.9.x) in place raises `RuntimeError: operator torchvision::nms does not exist` as soon as anything imports it.

In [ ]:
# Pinned after the other installs above so nothing pulls torch back to
# Colab's preinstalled 2.9.x, which dropped torchaudio.info(). torchvision is
# pinned in the same command so pip resolves all three together instead of
# potentially reinstalling a torchvision built against a different torch.
!pip install -q "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0"

**Patch `train.py` for the current `piper-sample-generator` API.** Verified against the package's source on GitHub/PyPI (v3.2.0): `generate_samples()` now lives in `piper_sample_generator.__main__` (not a top-level `generate_samples` module) and requires an explicit `model=` path instead of defaulting to one. Without this, Step 1 fails with `ModuleNotFoundError: No module named 'generate_samples'`, and fixing just the import would then fail with a missing `model` argument.

In [ ]:
train_py_path = "openwakeword/openwakeword/train.py"
with open(train_py_path) as f:
    train_py = f.read()

if "from piper_sample_generator.__main__ import generate_samples" not in train_py:
    train_py = train_py.replace(
        "from generate_samples import generate_samples",
        "from piper_sample_generator.__main__ import generate_samples",
    )
    train_py = train_py.replace(
        "generate_samples(",
        'generate_samples(model=os.path.join(config["piper_sample_generator_path"], "models", "en_US-libritts_r-medium.pt"), ',
    )
    with open(train_py_path, "w") as f:
        f.write(train_py)
    print("Patched", train_py_path)
else:
    print("Already patched, skipping")


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
rir_dataset = rir_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Convert audioset files to 16khz sample rate
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["mendy"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 10000       # yaml's own recommended minimum (notebook demo used 1000)
config["n_samples_val"] = 2000
config["steps"] = 50000           # yaml's own default (notebook demo used 10000)
# NOTE: dropped target_accuracy/target_recall from the original notebook here --
# verified these are dead keys, not read anywhere in the current train.py; quality
# is actually controlled by max_negative_weight / target_false_positives_per_hour,
# left at the yaml defaults below.

config["output_dir"] = "/content/drive/MyDrive/mendy_wake_word/my_custom_model"
import os
os.makedirs(config["output_dir"], exist_ok=True)

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Resample synthetic positive clips to 16kHz.
# piper-sample-generator==3.2.0 writes at the voice's native rate (22050Hz for
# en_US-libritts_r-medium.pt, per its .pt.json), not 16000Hz. openwakeword's
# augment_clips() hard-requires 16000Hz and raises "Error! Clip does not have
# the correct sample rate!" on the first mismatched clip -- which happens on
# the very first compute_features_from_generator call (positive train), so
# Step 2 dies before ever reaching the test-set features. The old dscripka
# piper fork emitted 16kHz directly; the stock PyPI package (installed above)
# doesn't, and generate_samples() exposes no sample_rate param to fix this at
# generation time, so clips are resampled in place here instead.
import soundfile as sf
import scipy.signal
import numpy as np

base = f'{config["output_dir"]}/{config["model_name"]}'
for sub in ("positive_train", "positive_test"):
    d = os.path.join(base, sub)
    for fn in os.listdir(d):
        if not fn.endswith(".wav"):
            continue
        p = os.path.join(d, fn)
        x, sr = sf.read(p)
        if sr == 16000:
            continue
        y = scipy.signal.resample_poly(x, 16000, sr)
        sf.write(p, (np.clip(y, -1, 1) * 32767).astype(np.int16), 16000, subtype="PCM_16")
print("Resampled any non-16kHz positive clips to 16kHz.")

In [ ]:
# Step 2: Augment the generated clips
# --overwrite forces train.py to recompute both feature files even if a stale
# positive_features_train.npy survives in the Drive-mounted output_dir from an
# earlier interrupted run (its skip-check only looks for that one file, not
# the _test.npy sibling, so a partial leftover silently skips regenerating both).

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips --overwrite

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!

In [ ]:
# Download a local copy too (the Drive copy at config['output_dir']
# already persisted -- this is just for convenience)
from google.colab import files
files.download(f"{config['output_dir']}/{config['model_name']}.onnx")


In [ ]:
# Quick self-test, mirroring exactly how wake_word_daemon.py loads the model
# (inference_framework="onnx", same Model.predict() call) -- upload a short WAV of
# someone saying "Mendy" first (left sidebar -> Files -> upload), then run this.
from openwakeword.model import Model
import scipy.io.wavfile as wavfile
import numpy as np

model = Model(wakeword_models=[f"{config['output_dir']}/{config['model_name']}.onnx"], inference_framework="onnx")

sr, audio = wavfile.read("test_mendy.wav")  # replace with your uploaded filename
assert sr == 16000, "must be 16kHz mono, matching wake_word_daemon.py's SAMPLE_RATE"
chunk = 1280  # 80ms at 16kHz, same as wake_word_daemon.py's CHUNK_SAMPLES
scores = [model.predict(audio[i:i+chunk])[config['model_name']]
          for i in range(0, len(audio) - chunk, chunk) if len(audio[i:i+chunk]) == chunk]
print("max score:", max(scores) if scores else "no full chunks")
